# Sentinel-1 / Sentinel-2 Co-collected Comparison Figure

Builds a figure with one row per date, each row showing a Sentinel-1 image (left) and a Sentinel-2 image (right).

**To change which dates appear, just edit the `DATES` list in the Config cell below** — everything else updates automatically.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import rasterio
import rasterio.mask
from rasterio.transform import array_bounds
import geopandas as gpd
from pyproj import Transformer
from shapely import wkt as shapely_wkt
from shapely.geometry import mapping
from shapely.ops import transform as shapely_transform
import matplotlib.pyplot as plt


## Config

- `DATES`: the list of dates you want, one per row, in `YYYYMMDD` format (matches the date string in your filenames). Reorder, add, or remove entries to change the figure — the number of rows follows the length of this list.
- `S2_FOLDER` / `S1_FOLDER`: folders to search (searched recursively).
- `S2_PATTERN` / `S1_PATTERN`: glob patterns with `{date}` as a placeholder, plus `*` wildcards for the parts of the filename that vary (acquisition time, orbit numbers, satellite A/B, etc.).

In [ ]:
# --- EDIT THIS to change which dates/rows are plotted ---
DATES = [
    "20230309",
    "20241110",
    "20251123"
]

# --- Folders and filename patterns ---
S2_FOLDER = "/home/jovyan/s2/mosaics_2022_2025_reflectance"
S2_PATTERN = "mosaic_{date}T*_clipped_BOA_reflectance.tif"

S1_FOLDER = "/home/jovyan/s1/preprocessed"
S1_PATTERN = "S1*_{date}T*_pre.tif"

# --- Sentinel-2 display bands (1-indexed, rasterio convention) ---
# Default assumes band order B2,B3,B4,... (Blue,Green,Red,...) -> RGB = (3,2,1)
# If colors look wrong/inverted, check band count/order printed below and adjust.
S2_RGB_BANDS = (3, 2, 1)

# --- Sentinel-1 display band(s) (1-indexed) ---
# Single int -> grayscale display of that band (e.g. VV).
# Tuple of 3 -> RGB-style composite (e.g. (VV, VH, VV) or (VV, VH, VV/VH) if you precompute a ratio band).
S1_DISPLAY_BANDS = 1

# --- Stretch percentiles for visualization ---
LOWER_PCT, UPPER_PCT = 2, 98

# --- Ocean masking (Sentinel-2 only) ---
# This figure focuses on the ocean, so land is masked out before stretching.
# Masking uses NDWI = (Green - NIR) / (Green + NIR); water is typically NDWI > 0.
# Set the 1-indexed band numbers for Green and NIR in your S2 stack (check band
# count/order in the sanity-check cell below if unsure -- commonly Green=2, NIR=4
# for a B2,B3,B4,B8 stack).
S2_GREEN_BAND = 2
S2_NIR_BAND = 4
OCEAN_NDWI_THRESHOLD = 0.0  # raise if too much land leaks in; lower if ocean edges get cut off

# Color used for masked-out land in the figure (matplotlib color name or hex)
LAND_BACKGROUND_COLOR = "black"

# --- Figure options ---
FIGSIZE_PER_ROW = 4.5  # inches of height per row
FIG_WIDTH = 9          # inches of total width (2 columns)
TITLE_FONTSIZE = 12

# --- Matched-pair acquisition times (for readable titles + time-difference) ---
# Path to the reviewed pairs CSV (Index, S1_Time, S2_Time, Time_Difference, ...).
# Rows are matched to DATES by the calendar date of S1_Time (YYYYMMDD).
MATCHED_PAIRS_CSV = "matched_pairs_reviewed.csv"

# --- Outflow locations (small red points + labels) ---
OUTFLOW_SHP = "/home/jovyan/s2/Outflow.shp"   # NOTE: assumed this is what you meant by "home/joyvan/s2/Outflow.shp"
OUTFLOW_LABEL_FIELD = "featureNam"
OUTFLOW_POINT_COLOR = "red"
OUTFLOW_POINT_SIZE = 12
OUTFLOW_LABEL_FONTSIZE = 7

# --- Simple lat/lon grid ---
SHOW_LATLON_GRID = True
LATLON_GRID_TICKS = 5       # number of gridlines per axis
LATLON_GRID_COLOR = "white"
LATLON_GRID_ALPHA = 0.5

# --- Sentinel-1 clip extent (AOI) ---
# S1 is clipped to this polygon before display. Given in lon/lat (EPSG:4326);
# it's reprojected on the fly to whatever CRS the S1 raster is actually in.
S1_CLIP_ENABLED = True
S1_CLIP_WKT = "POLYGON ((-117.35 32.4, -117.08 32.4, -117.08 32.75, -117.35 32.75, -117.35 32.4))"


## Helper functions

In [ ]:
def find_file(folder, pattern, date):
    """Find exactly one file in `folder` (recursively) matching `pattern` for `date`."""
    search = os.path.join(folder, pattern.format(date=date))
    matches = glob.glob(search, recursive=True)
    if len(matches) == 0:
        raise FileNotFoundError(f"No file found for date {date} matching: {search}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple files found for date {date}, expected exactly one:\n"
            + "\n".join(matches)
        )
    return matches[0]


def percentile_stretch(arr, lower=LOWER_PCT, upper=UPPER_PCT, stat_mask=None):
    """Stretch a single band to 0-1 using percentile clipping.

    NaNs/nodata are always ignored. If `stat_mask` is given, the lo/hi
    percentiles are computed only from pixels where stat_mask is True (e.g.
    ocean pixels), but the clip/normalize is still applied to every pixel --
    this is what lets the stretch be tuned to the ocean's dynamic range
    without those statistics being skewed by (usually much brighter) land.
    """
    arr = arr.astype("float32")
    valid = np.isfinite(arr)
    stat_pixels = valid if stat_mask is None else (valid & stat_mask)
    if not stat_pixels.any():
        stat_pixels = valid
    if not stat_pixels.any():
        return np.zeros_like(arr)
    lo, hi = np.nanpercentile(arr[stat_pixels], [lower, upper])
    if hi <= lo:
        hi = lo + 1e-6
    stretched = np.clip((arr - lo) / (hi - lo), 0, 1)
    stretched[~valid] = 0
    return stretched


def compute_ocean_mask(path, green_band=S2_GREEN_BAND, nir_band=S2_NIR_BAND,
                        threshold=OCEAN_NDWI_THRESHOLD):
    """Return a boolean array (True = ocean/water) using NDWI on the S2 file at `path`.

    NDWI = (Green - NIR) / (Green + NIR). Water reflects much less NIR than
    land/vegetation, so water pixels have higher (usually positive) NDWI.
    """
    with rasterio.open(path) as src:
        if max(green_band, nir_band) > src.count:
            raise ValueError(
                f"{os.path.basename(path)} only has {src.count} bands, but "
                f"S2_GREEN_BAND={green_band} / S2_NIR_BAND={nir_band} were requested. "
                "Adjust those in Config."
            )
        green = src.read(green_band).astype("float32")
        nir = src.read(nir_band).astype("float32")
        nodata = src.nodata
    if nodata is not None:
        green[green == nodata] = np.nan
        nir[nir == nodata] = np.nan
    denom = green + nir
    with np.errstate(invalid="ignore", divide="ignore"):
        ndwi = np.where(denom != 0, (green - nir) / denom, np.nan)
    ocean_mask = ndwi > threshold
    return ocean_mask


def load_s2_ocean_rgb(path, bands=S2_RGB_BANDS, green_band=S2_GREEN_BAND,
                       nir_band=S2_NIR_BAND, threshold=OCEAN_NDWI_THRESHOLD):
    """Load an RGBA ocean-only composite from a Sentinel-2 file.

    Land is masked out (fully transparent, alpha=0) using an NDWI water mask.
    The percentile stretch for each RGB channel is computed from ocean pixels
    only, so contrast is optimized for the water rather than being washed out
    or skewed by brighter land.

    Returns
    -------
    rgba : (H, W, 4) array
    crs  : the raster's CRS (needed to reproject the outflow points / build the lat-lon grid)
    extent : (left, right, bottom, top) in the raster's CRS units, for imshow(extent=...)
    """
    with rasterio.open(path) as src:
        if max(bands) > src.count:
            raise ValueError(
                f"{os.path.basename(path)} only has {src.count} bands, "
                f"but S2_RGB_BANDS={bands} was requested. Adjust S2_RGB_BANDS in Config."
            )
        data = src.read(list(bands)).astype("float32")
        nodata = src.nodata
        crs = src.crs
        b = src.bounds
        extent = (b.left, b.right, b.bottom, b.top)
    if nodata is not None:
        data[data == nodata] = np.nan

    ocean_mask = compute_ocean_mask(path, green_band=green_band, nir_band=nir_band,
                                     threshold=threshold)

    channels = [percentile_stretch(data[i], stat_mask=ocean_mask) for i in range(data.shape[0])]
    rgb = np.dstack(channels)

    alpha = np.where(ocean_mask, 1.0, 0.0).astype("float32")
    rgba = np.dstack([rgb, alpha])
    return rgba, crs, extent


def get_clip_geometry_for_crs(dst_crs, wkt=S1_CLIP_WKT, src_crs="EPSG:4326"):
    """Load the AOI polygon (given in lon/lat) and reproject it into `dst_crs`."""
    geom = shapely_wkt.loads(wkt)
    if str(dst_crs) != src_crs:
        transformer = Transformer.from_crs(src_crs, dst_crs, always_xy=True)
        geom = shapely_transform(transformer.transform, geom)
    return geom


def load_s1_display(path, bands=S1_DISPLAY_BANDS, clip=S1_CLIP_ENABLED, clip_wkt=S1_CLIP_WKT):
    """Load Sentinel-1 data for display: grayscale (single band) or composite (3 bands).

    If `clip` is True, the raster is first clipped to `clip_wkt` (an AOI polygon
    given in lon/lat, reprojected on the fly to the raster's own CRS).

    Returns
    -------
    img : 2D (grayscale) or (H, W, 3) array
    cmap : "gray" for single-band display, None for a 3-band composite
    crs  : the raster's CRS
    extent : (left, right, bottom, top) in the raster's CRS units, for imshow(extent=...)
    """
    with rasterio.open(path) as src:
        n = src.count
        crs = src.crs
        nodata = src.nodata
        band_list = [bands] if isinstance(bands, int) else list(bands)
        if max(band_list) > n:
            raise ValueError(
                f"{os.path.basename(path)} only has {n} bands, "
                f"but S1_DISPLAY_BANDS={bands} was requested."
            )

        if clip:
            clip_geom = get_clip_geometry_for_crs(crs, wkt=clip_wkt)
            data, transform = rasterio.mask.mask(
                src, [mapping(clip_geom)], crop=True, indexes=band_list,
                filled=True, nodata=nodata if nodata is not None else np.nan,
            )
            data = data.astype("float32")
            height, width = data.shape[1], data.shape[2]
            left, bottom, right, top = array_bounds(height, width, transform)
            extent = (left, right, bottom, top)
        else:
            data = src.read(band_list).astype("float32")
            b = src.bounds
            extent = (b.left, b.right, b.bottom, b.top)

        if nodata is not None:
            data[data == nodata] = np.nan

    if isinstance(bands, int):
        return percentile_stretch(data[0]), "gray", crs, extent
    else:
        channels = [percentile_stretch(data[i]) for i in range(data.shape[0])]
        return np.dstack(channels), None, crs, extent


## Outflow points + lat/lon grid helpers

- `add_outflow_points` reprojects the `Outflow.shp` points into whichever CRS the current raster is in, then plots the ones that fall inside the image as small red dots labeled from the `featureNam` field.
- `add_latlon_grid` keeps the gridline *positions* in the raster's native (projected) CRS, but converts the *tick labels* to lat/lon -- this gives a simple, readable lat/lon grid on any CRS without needing a full reprojected graticule.


In [ ]:
_outflow_gdf_cache = None

def load_outflow_points(path=OUTFLOW_SHP):
    """Load the outflow-location shapefile once and cache it (in its native CRS)."""
    global _outflow_gdf_cache
    if _outflow_gdf_cache is None:
        _outflow_gdf_cache = gpd.read_file(path)
    return _outflow_gdf_cache


def add_outflow_points(ax, raster_crs, extent, shp_path=OUTFLOW_SHP,
                        label_field=OUTFLOW_LABEL_FIELD,
                        color=OUTFLOW_POINT_COLOR, size=OUTFLOW_POINT_SIZE,
                        label_fontsize=OUTFLOW_LABEL_FONTSIZE):
    """Plot outflow points (small red dots) + their `featureNam` labels on `ax`.

    Points are reprojected to match the raster's CRS, and only points that
    fall inside `extent` (the current image's bounds) are drawn/labeled.
    """
    gdf = load_outflow_points(shp_path)
    gdf_proj = gdf.to_crs(raster_crs) if gdf.crs is not None else gdf

    left, right, bottom, top = extent
    for _, feat in gdf_proj.iterrows():
        geom = feat.geometry
        if geom is None or geom.is_empty:
            continue
        x, y = geom.x, geom.y
        if not (left <= x <= right and bottom <= y <= top):
            continue
        ax.scatter(x, y, s=size, c=color, marker="o",
                   edgecolors="white", linewidths=0.3, zorder=5)
        label = feat.get(label_field, "")
        if label:
            ax.annotate(str(label), (x, y), xytext=(3, 3),
                        textcoords="offset points", fontsize=label_fontsize,
                        color=color, zorder=6)


def format_lon(lon):
    hemi = "E" if lon >= 0 else "W"
    return f"{abs(lon):.2f}\u00b0{hemi}"


def format_lat(lat):
    hemi = "N" if lat >= 0 else "S"
    return f"{abs(lat):.2f}\u00b0{hemi}"


def add_latlon_grid(ax, raster_crs, extent, n_ticks=LATLON_GRID_TICKS,
                     color=LATLON_GRID_COLOR, alpha=LATLON_GRID_ALPHA):
    """Draw a simple lat/lon tick grid on `ax` (tick positions stay in the
    raster's native CRS; only the tick labels are converted to lat/lon)."""
    left, right, bottom, top = extent
    transformer = Transformer.from_crs(raster_crs, "EPSG:4326", always_xy=True)

    xticks = np.linspace(left, right, n_ticks)
    yticks = np.linspace(bottom, top, n_ticks)
    ymid = (bottom + top) / 2
    xmid = (left + right) / 2

    lons, _ = transformer.transform(xticks, [ymid] * n_ticks)
    _, lats = transformer.transform([xmid] * n_ticks, yticks)

    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)
    ax.set_xticks(xticks)
    ax.set_xticklabels([format_lon(v) for v in lons], fontsize=7)
    ax.set_yticks(yticks)
    ax.set_yticklabels([format_lat(v) for v in lats], fontsize=7)
    ax.tick_params(axis="both", which="both", length=2)
    ax.grid(True, color=color, alpha=alpha, linewidth=0.6, linestyle="--")
    for spine in ax.spines.values():
        spine.set_edgecolor("gray")
        spine.set_linewidth(0.5)


## Matched-pair acquisition times (from `matched_pairs_reviewed.csv`)

Looks up each date's row in the reviewed-pairs CSV to get the exact S1/S2 acquisition timestamps and format them nicely, plus compute the time difference between the pair.


In [ ]:
def load_matched_pairs(csv_path=MATCHED_PAIRS_CSV):
    """Load the CSV and index it by the S1 acquisition date (YYYYMMDD) so it
    can be looked up by the same date strings used in DATES."""
    df = pd.read_csv(csv_path)
    df["S1_Time"] = pd.to_datetime(df["S1_Time"], utc=True)
    df["S2_Time"] = pd.to_datetime(df["S2_Time"], utc=True)
    df["_date_key"] = df["S1_Time"].dt.strftime("%Y%m%d")
    return df.set_index("_date_key")


def format_dt(dt):
    """e.g. '09 Mar 2023, 13:44 UTC'"""
    return dt.strftime("%d %b %Y, %H:%M UTC")


def format_timediff(td):
    """Format a pandas Timedelta as e.g. '4h 53m'."""
    total_minutes = int(round(td.total_seconds() / 60))
    hours, minutes = divmod(total_minutes, 60)
    if hours and minutes:
        return f"{hours}h {minutes}m"
    if hours:
        return f"{hours}h"
    return f"{minutes}m"


## (Optional) sanity-check available files and band counts

Run this to confirm each date resolves to exactly one file, and to check how many bands each file has (useful for setting `S2_RGB_BANDS` / `S1_DISPLAY_BANDS` correctly).

In [4]:
for date in DATES:
    s2_path = find_file(S2_FOLDER, S2_PATTERN, date)
    s1_path = find_file(S1_FOLDER, S1_PATTERN, date)
    with rasterio.open(s2_path) as src:
        s2_bands = src.count
    with rasterio.open(s1_path) as src:
        s1_bands = src.count
    print(f"{date}:")
    print(f"  S2 -> {s2_path}  ({s2_bands} bands)")
    print(f"  S1 -> {s1_path}  ({s1_bands} bands)")

20230309:
  S2 -> /home/jovyan/s2/mosaics_2022_2025_reflectance/mosaic_20230309T183209_clipped_BOA_reflectance.tif  (11 bands)
  S1 -> /home/jovyan/s1/preprocessed/S1A_IW_GRDH_1SDV_20230309T135313_20230309T135338_047568_05B656_247B_pre.tif  (2 bands)
20241110:
  S2 -> /home/jovyan/s2/mosaics_2022_2025_reflectance/mosaic_20241110T182601_clipped_BOA_reflectance.tif  (11 bands)
  S1 -> /home/jovyan/s1/preprocessed/S1A_IW_GRDH_1SDV_20241110T135319_20241110T135344_056493_06ECD4_BBBF_pre.tif  (2 bands)
20251123:
  S2 -> /home/jovyan/s2/mosaics_2022_2025_reflectance/mosaic_20251123T183609_clipped_BOA_reflectance.tif  (11 bands)
  S1 -> /home/jovyan/s1/preprocessed/S1C_IW_GRDH_1SDV_20251123T135159_20251123T135224_005142_00A328_B231_pre.tif  (2 bands)


## Build the figure

In [ ]:
n_rows = len(DATES)
fig, axes = plt.subplots(
    n_rows, 2, figsize=(FIG_WIDTH, FIGSIZE_PER_ROW * n_rows)
)
if n_rows == 1:
    axes = axes.reshape(1, 2)

matched_pairs = load_matched_pairs()

for row, date in enumerate(DATES):
    s1_path = find_file(S1_FOLDER, S1_PATTERN, date)
    s2_path = find_file(S2_FOLDER, S2_PATTERN, date)

    s1_img, s1_cmap, s1_crs, s1_extent = load_s1_display(s1_path)
    s2_img, s2_crs, s2_extent = load_s2_ocean_rgb(s2_path)

    ax_s1, ax_s2 = axes[row, 0], axes[row, 1]

    # --- readable datetime titles + time difference between the pair, from the CSV ---
    if date in matched_pairs.index:
        pair = matched_pairs.loc[date]
        s1_dt, s2_dt = pair["S1_Time"], pair["S2_Time"]
        diff_str = format_timediff(abs(s2_dt - s1_dt))
        s1_title = f"Sentinel-1 — {format_dt(s1_dt)}"
        s2_title = f"Sentinel-2 (ocean only) — {format_dt(s2_dt)}\nΔt from S1: {diff_str}"
    else:
        print(f"Warning: no matching row in {MATCHED_PAIRS_CSV} for date {date}")
        s1_title = f"Sentinel-1 — {date}"
        s2_title = f"Sentinel-2 (ocean only) — {date}"

    ax_s1.imshow(s1_img, cmap=s1_cmap, extent=s1_extent, origin="upper")
    ax_s1.set_title(s1_title, fontsize=TITLE_FONTSIZE)
    ax_s1.set_aspect("equal")

    ax_s2.set_facecolor(LAND_BACKGROUND_COLOR)
    ax_s2.imshow(s2_img, extent=s2_extent, origin="upper")
    ax_s2.set_title(s2_title, fontsize=TITLE_FONTSIZE)
    ax_s2.set_aspect("equal")

    # --- simple lat/lon grid (labels only; keeps positions in the native CRS) ---
    if SHOW_LATLON_GRID:
        add_latlon_grid(ax_s1, s1_crs, s1_extent)
        add_latlon_grid(ax_s2, s2_crs, s2_extent)
    else:
        ax_s1.axis("off")
        ax_s2.axis("off")

    # --- outflow locations (small red points + featureNam labels) ---
    add_outflow_points(ax_s1, s1_crs, s1_extent)
    add_outflow_points(ax_s2, s2_crs, s2_extent)

plt.tight_layout()
plt.savefig("s1_s2_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
